# TER — Variation du Weight Decay sur BERT / SNLI

Ce notebook compare l'effet de plusieurs valeurs de **weight decay** sur un même modèle **BERT fine-tuné sur SNLI**.

L'objectif est de montrer que la régularisation influence :
- les performances de validation sur SNLI ;
- la stabilité de l'apprentissage ;
- la capacité de généralisation sur des jeux hors-distribution ;
- la robustesse face aux heuristiques syntaxiques avec HANS.

Valeurs testées : `0`, `0.01`, `0.05`, `0.1`.

Le protocole garde tous les autres paramètres constants afin d'isoler l'effet du weight decay.
 — version GitHub propre

In [ ]:
# Installation des bibliothèques nécessaires
!pip install -q transformers[torch] datasets evaluate accelerate optuna scikit-learn matplotlib pandas seaborn captum

# Vérification du GPU
import torch
print(f"GPU détecté : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN GPU'}")

In [ ]:
# ============================================================
# 2. Imports
# ============================================================
import os
import gc
import json
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
# ============================================================
# 3. Connexion à Google Drive
# ============================================================
# Option Colab : décommentez si vous souhaitez monter Google Drive.


In [ ]:
# ============================================================
# 4. Configuration générale
# ============================================================

MODEL_NAME = "bert-base-uncased"

# Hyperparamètre étudié
WEIGHT_DECAYS = [0.0, 0.01, 0.05, 0.1]

# Hyperparamètres fixés pour isoler l'effet du weight decay
LEARNING_RATE = 2e-5
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 2
SEED = 42

# Échantillons
# Pour une analyse solide : 50_000 / 5_000
TRAIN_SIZE = 50_000
VALID_SIZE = 5_000

# Échantillons d'évaluation hors-distribution
# Tu peux augmenter ces valeurs si tu veux une analyse plus complète.
HANS_EVAL_SIZE = 10_000
ANLI_EVAL_SIZE = 1_200

# Dossiers de sauvegarde
BASE_DIR = os.environ.get("TER_OUTPUT_DIR", ".")
SAVE_DIR = f"{BASE_DIR}/resultats_weight_decay"
MODEL_OUTPUT_DIR = f"{SAVE_DIR}/checkpoints"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))

print("Dossier de sauvegarde :", SAVE_DIR)

In [ ]:
# ============================================================
# 5. Chargement du tokenizer
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# ============================================================
# 6. Chargement et préparation de SNLI
# ============================================================

print("Chargement de SNLI...")

snli = load_dataset("snli")

# On filtre les labels -1, qui correspondent à des exemples sans consensus.
snli_train_full = snli["train"].filter(lambda x: x["label"] != -1)
snli_validation_full = snli["validation"].filter(lambda x: x["label"] != -1)

train_dataset = snli_train_full.shuffle(seed=SEED).select(range(TRAIN_SIZE))
validation_dataset = snli_validation_full.shuffle(seed=SEED).select(range(VALID_SIZE))

print(train_dataset)
print(validation_dataset)

In [ ]:
# ============================================================
# 7. Prétraitement / tokenisation SNLI
# ============================================================

def preprocess_nli(examples):
    return tokenizer(
        examples["premise"],
        examples["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

print("Tokenisation SNLI en cours...")

tokenized_train = train_dataset.map(preprocess_nli, batched=True)
tokenized_validation = validation_dataset.map(preprocess_nli, batched=True)

cols_to_remove_train = [col for col in ["premise", "hypothesis"] if col in tokenized_train.column_names]
cols_to_remove_val = [col for col in ["premise", "hypothesis"] if col in tokenized_validation.column_names]

tokenized_train = tokenized_train.remove_columns(cols_to_remove_train)
tokenized_validation = tokenized_validation.remove_columns(cols_to_remove_val)

tokenized_train.set_format("torch")
tokenized_validation.set_format("torch")

print("Tokenisation SNLI terminée.")
print(tokenized_train)
print(tokenized_validation)

In [ ]:
# ============================================================
# 8. Fonction de métriques multiclasse
# ============================================================

LABEL_NAMES = ["entailment", "neutral", "contradiction"]

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "macro_f1": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "weighted_f1": f1_weighted,
    }

## Préparation des jeux d'évaluation hors-distribution

L'évaluation sur SNLI mesure la performance dans un environnement proche de l'entraînement.

Pour étudier la robustesse, le notebook ajoute :
- **HANS**, pour mesurer la sensibilité aux heuristiques syntaxiques ;
- **ANLI**, pour mesurer la robustesse face à des exemples adversariaux.

Dans HANS, les labels sont binaires : `entailment` et `non-entailment`.  
Comme le modèle prédit trois classes NLI, on regroupe `neutral` et `contradiction` en `non-entailment`.


In [ ]:
# ============================================================
# 9. Chargement et préparation de HANS
# ============================================================

print("Chargement de HANS...")
hans = load_dataset("hans", split="validation" ,revision="refs/convert/parquet")

if HANS_EVAL_SIZE is not None and HANS_EVAL_SIZE < len(hans):
    hans = hans.shuffle(seed=SEED).select(range(HANS_EVAL_SIZE))

def encode_hans_label(example):
    # HANS : entailment / non-entailment
    # On code entailment = 0 ; non-entailment = 1 pour l'évaluation binaire.
    example["binary_label"] = 0 if example["label"] == "entailment" else 1
    return example

hans = hans.map(encode_hans_label)

tokenized_hans = hans.map(preprocess_nli, batched=True)
cols_to_remove_hans = [
    col for col in tokenized_hans.column_names
    if col not in ["input_ids", "token_type_ids", "attention_mask", "binary_label", "heuristic"]
]

tokenized_hans = tokenized_hans.remove_columns(cols_to_remove_hans)
tokenized_hans.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "binary_label"])

print(hans)
print(tokenized_hans)

In [ ]:
# ============================================================
# 10. Chargement et préparation de ANLI
# ============================================================

print("Chargement de ANLI...")

anli = load_dataset("anli")

ANLI_SPLITS = {
    "ANLI_R1": "dev_r1",
    "ANLI_R2": "dev_r2",
    "ANLI_R3": "dev_r3",
}

tokenized_anli = {}

for dataset_name, split_name in ANLI_SPLITS.items():
    ds = anli[split_name].filter(lambda x: x["label"] != -1)

    if ANLI_EVAL_SIZE is not None and ANLI_EVAL_SIZE < len(ds):
        ds = ds.shuffle(seed=SEED).select(range(ANLI_EVAL_SIZE))

    tok = ds.map(preprocess_nli, batched=True)
    cols_to_remove = [col for col in ["premise", "hypothesis", "reason", "uid"] if col in tok.column_names]
    tok = tok.remove_columns(cols_to_remove)
    tok.set_format("torch")

    tokenized_anli[dataset_name] = tok
    print(dataset_name, tok)

In [ ]:
# ============================================================
# 11. Fonctions d'évaluation complémentaires
# ============================================================

def evaluate_multiclass_dataset(trainer, dataset, dataset_name):
    metrics = trainer.evaluate(eval_dataset=dataset)
    return {
        "dataset": dataset_name,
        "eval_loss": metrics.get("eval_loss"),
        "accuracy": metrics.get("eval_accuracy"),
        "macro_f1": metrics.get("eval_macro_f1"),
        "weighted_f1": metrics.get("eval_weighted_f1"),
    }


def evaluate_hans_binary(trainer, tokenized_hans_dataset, original_hans_dataset):
    predictions_output = trainer.predict(tokenized_hans_dataset)
    logits = predictions_output.predictions

    preds_3classes = np.argmax(logits, axis=-1)

    # BERT SNLI : 0 = entailment, 1 = neutral, 2 = contradiction
    # Pour HANS : neutral + contradiction => non-entailment
    preds_binary = np.where(preds_3classes == 0, 0, 1)

    y_true = np.array(original_hans_dataset["binary_label"])

    hans_accuracy = accuracy_score(y_true, preds_binary)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true,
        preds_binary,
        average="macro",
        zero_division=0
    )

    # Rappel spécifique de la classe non-entailment : utile pour mesurer le biais de prédiction entailment.
    precision_by_class, recall_by_class, f1_by_class, support_by_class = precision_recall_fscore_support(
        y_true,
        preds_binary,
        labels=[0, 1],
        zero_division=0
    )

    return {
        "dataset": "HANS",
        "eval_loss": None,
        "accuracy": hans_accuracy,
        "macro_f1": f1_macro,
        "weighted_f1": None,
        "hans_entailment_recall": recall_by_class[0],
        "hans_non_entailment_recall": recall_by_class[1],
    }, y_true, preds_binary

In [ ]:
# ============================================================
# 12. Boucle d'entraînement avec variation du weight decay
# ============================================================

resultats_validation = []
resultats_robustesse = []
historiques = []
matrices_hans = {}

for wd in WEIGHT_DECAYS:
    print("====================================")
    print(f"Entraînement BERT avec weight decay = {wd}")
    print("====================================")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # On recrée un modèle neuf pour chaque valeur de weight decay.
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3
    )

    safe_wd_name = str(wd).replace(".", "_")
    run_dir = f"{MODEL_OUTPUT_DIR}/bert_wd_{safe_wd_name}"

    training_args = TrainingArguments(
        output_dir=run_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=wd,
        logging_dir=f"{SAVE_DIR}/logs_bert_wd_{safe_wd_name}",
        logging_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
        seed=SEED
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_validation,
        compute_metrics=compute_metrics
    )

    train_output = trainer.train()
    eval_output = trainer.evaluate(eval_dataset=tokenized_validation)

    resultats_validation.append({
        "weight_decay": wd,
        "train_loss": train_output.metrics["train_loss"],
        "eval_loss": eval_output["eval_loss"],
        "accuracy": eval_output["eval_accuracy"],
        "precision_macro": eval_output["eval_precision_macro"],
        "recall_macro": eval_output["eval_recall_macro"],
        "macro_f1": eval_output["eval_macro_f1"],
        "precision_weighted": eval_output["eval_precision_weighted"],
        "recall_weighted": eval_output["eval_recall_weighted"],
        "weighted_f1": eval_output["eval_weighted_f1"],
        "train_runtime": train_output.metrics["train_runtime"],
        "train_samples_per_second": train_output.metrics["train_samples_per_second"]
    })

    # Historique par époque pour les courbes
    for item in trainer.state.log_history:
        row = item.copy()
        row["weight_decay"] = wd
        historiques.append(row)

    # Évaluation HANS
    hans_metrics, hans_y_true, hans_y_pred = evaluate_hans_binary(
        trainer,
        tokenized_hans,
        hans
    )
    hans_metrics["weight_decay"] = wd
    resultats_robustesse.append(hans_metrics)
    matrices_hans[wd] = (hans_y_true, hans_y_pred)

    # Évaluation ANLI R1/R2/R3
    for dataset_name, tok_ds in tokenized_anli.items():
        anli_metrics = evaluate_multiclass_dataset(trainer, tok_ds, dataset_name)
        anli_metrics["weight_decay"] = wd
        resultats_robustesse.append(anli_metrics)

    # Nettoyage mémoire GPU après chaque entraînement
    del model
    del trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Tous les entraînements sont terminés.")

In [ ]:
# ============================================================
# 13. Tableaux des résultats
# ============================================================

df_validation = pd.DataFrame(resultats_validation).sort_values(
    by="macro_f1",
    ascending=False
)

df_robustesse = pd.DataFrame(resultats_robustesse)

df_historique = pd.DataFrame(historiques)

display(df_validation)
display(df_robustesse)
display(df_historique.head())

In [ ]:
# ============================================================
# 14. Tableau simplifié pour le rapport
# ============================================================

df_resume_validation = df_validation[
    [
        "weight_decay",
        "train_loss",
        "eval_loss",
        "accuracy",
        "macro_f1",
        "weighted_f1",
        "train_runtime"
    ]
].copy()

df_resume_validation["weight_decay"] = df_resume_validation["weight_decay"].astype(str)

display(df_resume_validation)

In [ ]:
# ============================================================
# 15. Tableau pivot robustesse
# ============================================================

df_pivot_robustesse = df_robustesse.pivot_table(
    index="weight_decay",
    columns="dataset",
    values=["accuracy", "macro_f1"],
    aggfunc="first"
)

display(df_pivot_robustesse)

In [ ]:
# ============================================================
# 16. Graphique : performance SNLI validation selon le weight decay
# ============================================================

df_plot = df_validation.sort_values("weight_decay")

plt.figure(figsize=(10, 6))

plt.plot(
    df_plot["weight_decay"],
    df_plot["macro_f1"],
    marker="o",
    label="Macro-F1 SNLI validation"
)

plt.plot(
    df_plot["weight_decay"],
    df_plot["accuracy"],
    marker="o",
    label="Accuracy SNLI validation"
)

plt.xlabel("Weight Decay")
plt.ylabel("Score")
plt.title("Impact du Weight Decay sur les performances de validation SNLI")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# 17. Graphique : loss selon le weight decay
# ============================================================

df_plot = df_validation.sort_values("weight_decay")

plt.figure(figsize=(10, 6))

plt.plot(
    df_plot["weight_decay"],
    df_plot["train_loss"],
    marker="o",
    label="Train loss"
)

plt.plot(
    df_plot["weight_decay"],
    df_plot["eval_loss"],
    marker="o",
    label="Validation loss"
)

plt.xlabel("Weight Decay")
plt.ylabel("Loss")
plt.title("Impact du Weight Decay sur la loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# 18. Graphique : robustesse HANS selon le weight decay
# ============================================================

df_hans = df_robustesse[df_robustesse["dataset"] == "HANS"].sort_values("weight_decay")

plt.figure(figsize=(10, 6))

plt.plot(
    df_hans["weight_decay"],
    df_hans["accuracy"],
    marker="o",
    label="Accuracy HANS"
)

plt.plot(
    df_hans["weight_decay"],
    df_hans["macro_f1"],
    marker="o",
    label="Macro-F1 HANS"
)

plt.plot(
    df_hans["weight_decay"],
    df_hans["hans_non_entailment_recall"],
    marker="o",
    label="Recall Non-Entailment HANS"
)

plt.xlabel("Weight Decay")
plt.ylabel("Score")
plt.title("Impact du Weight Decay sur la robustesse syntaxique HANS")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# 19. Graphique : robustesse ANLI selon le weight decay
# ============================================================

df_anli = df_robustesse[df_robustesse["dataset"].str.startswith("ANLI")].copy()
df_anli = df_anli.sort_values(["dataset", "weight_decay"])

plt.figure(figsize=(10, 6))

for dataset_name in sorted(df_anli["dataset"].unique()):
    subset = df_anli[df_anli["dataset"] == dataset_name]
    plt.plot(
        subset["weight_decay"],
        subset["accuracy"],
        marker="o",
        label=f"{dataset_name} accuracy"
    )

plt.xlabel("Weight Decay")
plt.ylabel("Accuracy")
plt.title("Impact du Weight Decay sur la robustesse adversariale ANLI")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# 20. Courbes d'apprentissage par époque
# ============================================================

df_hist = df_historique.copy()

# Courbe des losses de validation si disponibles
df_eval_hist = df_hist[df_hist["eval_loss"].notna()].copy() if "eval_loss" in df_hist.columns else pd.DataFrame()

if not df_eval_hist.empty:
    plt.figure(figsize=(10, 6))

    for wd in sorted(df_eval_hist["weight_decay"].unique()):
        subset = df_eval_hist[df_eval_hist["weight_decay"] == wd]
        plt.plot(
            subset["epoch"],
            subset["eval_loss"],
            marker="o",
            label=f"WD={wd}"
        )

    plt.xlabel("Epoch")
    plt.ylabel("Validation loss")
    plt.title("Évolution de la validation loss selon le Weight Decay")
    plt.legend()
    plt.grid(True)
    plt.show()

# Courbe du macro-F1 de validation si disponible
if not df_eval_hist.empty and "eval_macro_f1" in df_eval_hist.columns:
    plt.figure(figsize=(10, 6))

    for wd in sorted(df_eval_hist["weight_decay"].unique()):
        subset = df_eval_hist[df_eval_hist["weight_decay"] == wd]
        plt.plot(
            subset["epoch"],
            subset["eval_macro_f1"],
            marker="o",
            label=f"WD={wd}"
        )

    plt.xlabel("Epoch")
    plt.ylabel("Macro-F1 validation")
    plt.title("Évolution du Macro-F1 selon le Weight Decay")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# ============================================================
# 21. Matrices de confusion HANS
# ============================================================

for wd, (y_true, y_pred) in matrices_hans.items():
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Entailment", "Non-entailment"]
    )

    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, values_format="d")
    plt.title(f"Matrice de confusion HANS — Weight Decay = {wd}")
    plt.show()

In [ ]:
# ============================================================
# 22. Sélection automatique des meilleurs réglages
# ============================================================

best_snli = df_validation.sort_values("macro_f1", ascending=False).iloc[0]

print("Meilleur weight decay selon le Macro-F1 SNLI validation :")
print(best_snli["weight_decay"])
print(best_snli)

if not df_hans.empty:
    best_hans = df_hans.sort_values("macro_f1", ascending=False).iloc[0]
    print("\nMeilleur weight decay selon le Macro-F1 HANS :")
    print(best_hans["weight_decay"])
    print(best_hans)

if not df_anli.empty:
    df_anli_mean = df_anli.groupby("weight_decay", as_index=False).agg(
        anli_accuracy_mean=("accuracy", "mean"),
        anli_macro_f1_mean=("macro_f1", "mean")
    )

    best_anli = df_anli_mean.sort_values("anli_macro_f1_mean", ascending=False).iloc[0]
    print("\nMeilleur weight decay selon le Macro-F1 moyen ANLI :")
    print(best_anli["weight_decay"])
    print(best_anli)

    display(df_anli_mean)

In [ ]:
# ============================================================
# 23. Export CSV et LaTeX
# ============================================================

df_validation.to_csv(f"{SAVE_DIR}/resultats_validation_weight_decay_bert.csv", index=False)
df_resume_validation.to_csv(f"{SAVE_DIR}/resume_validation_weight_decay_bert.csv", index=False)
df_robustesse.to_csv(f"{SAVE_DIR}/resultats_robustesse_weight_decay_bert.csv", index=False)
df_historique.to_csv(f"{SAVE_DIR}/historique_weight_decay_bert.csv", index=False)

latex_validation = df_resume_validation.to_latex(
    index=False,
    float_format="%.4f",
    caption="Impact du weight decay sur les performances de validation du modèle BERT entraîné sur SNLI",
    label="tab:weight_decay_bert_snli"
)

with open(f"{SAVE_DIR}/tableau_validation_weight_decay_bert.tex", "w", encoding="utf-8") as f:
    f.write(latex_validation)

# Tableau synthétique robustesse
df_robustesse_resume = df_robustesse[
    ["weight_decay", "dataset", "accuracy", "macro_f1"]
].copy()

latex_robustesse = df_robustesse_resume.to_latex(
    index=False,
    float_format="%.4f",
    caption="Impact du weight decay sur la robustesse hors-distribution du modèle BERT",
    label="tab:weight_decay_robustesse_bert"
)

with open(f"{SAVE_DIR}/tableau_robustesse_weight_decay_bert.tex", "w", encoding="utf-8") as f:
    f.write(latex_robustesse)

print("Fichiers sauvegardés dans :", SAVE_DIR)
print(os.listdir(SAVE_DIR))

## Interprétation à reprendre dans le rapport

Le weight decay agit comme une régularisation : il limite l'ampleur des poids appris pendant le fine-tuning.  
Dans le cadre du NLI, son intérêt ne se limite pas à améliorer le score sur SNLI. Il permet surtout d'étudier le compromis entre :

- **performance in-distribution**, mesurée sur SNLI validation ;
- **robustesse hors-distribution**, mesurée avec ANLI ;
- **résistance aux heuristiques syntaxiques**, mesurée avec HANS.

L'analyse doit donc vérifier si la meilleure valeur de weight decay sur SNLI est aussi celle qui donne les meilleurs résultats sur HANS ou ANLI.  
Si ce n'est pas le cas, cela montre qu'un modèle très performant sur le benchmark d'origine peut rester fragile face aux biais et aux exemples adversariaux.
